In [100]:
import pandas as pd

columns = ["user_id", "item_id", "rating", "timestamp"]

# Carregando os dados de avaliações
ratings = pd.read_csv(
    "../data/u.data",
    sep="\t",
    names=columns
) 

# Criando a matriz de usuário-item com os valores de avaliação
user_item_matrix = ratings.pivot(
    index="user_id",
    columns="item_id",
    values="rating"
)

user_item_matrix.head()


item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [101]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#criando a tabela de similaridade entre os usuários usando a similaridade do cosseno
user_item_filled = user_item_matrix.fillna(0)
user_similarity = cosine_similarity(user_item_filled)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_filled.index,
    columns=user_item_filled.index
)

user_similarity_df.head()


user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.166931,0.047460,0.064358,0.378475,0.430239,0.440367,0.319072,0.078138,0.376544,...,0.369527,0.119482,0.274876,0.189705,0.197326,0.118095,0.314072,0.148617,0.179508,0.398175
2,0.166931,1.000000,0.110591,0.178121,0.072979,0.245843,0.107328,0.103344,0.161048,0.159862,...,0.156986,0.307942,0.358789,0.424046,0.319889,0.228583,0.226790,0.161485,0.172268,0.105798
3,0.047460,0.110591,1.000000,0.344151,0.021245,0.072415,0.066137,0.083060,0.061040,0.065151,...,0.031875,0.042753,0.163829,0.069038,0.124245,0.026271,0.161890,0.101243,0.133416,0.026556
4,0.064358,0.178121,0.344151,1.000000,0.031804,0.068044,0.091230,0.188060,0.101284,0.060859,...,0.052107,0.036784,0.133115,0.193471,0.146058,0.030138,0.196858,0.152041,0.170086,0.058752
5,0.378475,0.072979,0.021245,0.031804,1.000000,0.237286,0.373600,0.248930,0.056847,0.201427,...,0.338794,0.080580,0.094924,0.079779,0.148607,0.071459,0.239955,0.139595,0.152497,0.313941


In [102]:
def recommend_items(
    user_id,
    user_item_matrix,
    user_similarity_df,
    k=5,
    n_recommendations=5
):
    # 1. Usuários mais similares (ignorando ele mesmo)
    similar_users = (
        user_similarity_df[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
        .head(k)
    )

    # 2. Itens que o usuário alvo já avaliou para não recomendar novamente
    user_ratings = user_item_matrix.loc[user_id]
    user_rated_items = user_ratings[user_ratings.notna()].index

    # 3. Notas dos usuários mais similares
    similar_users_ratings = user_item_matrix.loc[similar_users.index]

    # pesos = similaridade dos usuários (k vizinhos)
    weights = similar_users

    # soma ponderada das notas por item
    weighted_sum = similar_users_ratings.T.dot(weights)

    # normaliza pela soma dos pesos (evita divisão por zero)
    den = weights.sum()
    if den == 0:
        return pd.Series(dtype=float)

    mean_ratings = weighted_sum / den

    # remove itens sem nota (NaN) e já avaliados
    mean_ratings = mean_ratings.dropna()
    recomendacoes = mean_ratings.drop(user_rated_items, errors="ignore")

    # 6. Retorna os melhores
    return recomendacoes.sort_values(ascending=False).head(n_recommendations)

user_id = user_item_matrix.index[0]

recommend_items(
    user_id,
    user_item_matrix,
    user_similarity_df,
    k=5,
    n_recommendations=5
)



item_id
474    4.001246
273    3.990894
433    3.986334
566    3.594161
382    3.405275
dtype: float64

In [103]:
# ...existing code...
import numpy as np

# ===== Avaliação offline simples (Precision@K e Recall@K) =====
def train_test_split_per_user(ratings_df, test_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    test_idx = []

    for user_id, group in ratings_df.groupby("user_id"):
        n = len(group)
        if n < 2:
            continue
        test_size = max(1, int(n * test_ratio))
        test_indices = rng.choice(group.index, size=test_size, replace=False)
        test_idx.extend(test_indices)

    test_df = ratings_df.loc[test_idx]
    train_df = ratings_df.drop(test_idx)
    return train_df, test_df

def precision_recall_at_k(recommended, relevant, k):
    if k == 0:
        return 0.0, 0.0
    recommended_k = recommended[:k]
    hit_count = len(set(recommended_k) & set(relevant))
    precision = hit_count / k
    recall = hit_count / max(1, len(relevant))
    return precision, recall

# 1) Split
train_df, test_df = train_test_split_per_user(ratings, test_ratio=0.2, seed=42)

# 2) Recria matriz e similaridade com treino
train_matrix = train_df.pivot(index="user_id", columns="item_id", values="rating")
train_filled = train_matrix.fillna(0)
train_similarity = cosine_similarity(train_filled)
train_similarity_df = pd.DataFrame(
    train_similarity,
    index=train_filled.index,
    columns=train_filled.index
)

# 3) Avalia
K = 10
precisions = []
recalls = []

for user_id, group in test_df.groupby("user_id"):
    if user_id not in train_matrix.index:
        continue

    relevant_items = group[group["rating"] > 3]["item_id"].tolist()
    recs = recommend_items(
        user_id,
        train_matrix,
        train_similarity_df,
        k=5,
        n_recommendations=K
    )

    recommended_items = recs.index.tolist()
    p, r = precision_recall_at_k(recommended_items, relevant_items, K)
    precisions.append(p)
    recalls.append(r)

print(f"Precision@{K}: {np.mean(precisions):.4f}")
print(f"Recall@{K}: {np.mean(recalls):.4f}")
# ...existing code...

Precision@10: 0.0645
Recall@10: 0.0399
